# Binned Parametric Populations

A parametric `Stars` object stores its star formation and metal enrichment history as a set of bins, each holding the mass formed in it. The mass is spread uniformly within each bin, and a bin can have zero width (a single age or metallicity). When the emission is extracted the bins are spread onto the grid with a "bin in cell" (BIC) approach: the extension of the cloud in cell weights used for particles to bins of finite width. A zero width bin therefore gets exactly the cloud in cell weights a particle at that point would.

Because the bins are used exactly as given, they never need to match the grid. This makes it straightforward to use tabulated star formation histories, e.g. the output of a semi-analytic model (SAM). In this example we cover the two constructors for binned inputs, `Stars.from_sfzh` and `Stars.from_binned`, and how the SAM loaders use them.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from unyt import Msun, Myr, yr

from synthesizer.emission_models import IncidentEmission
from synthesizer.grid import Grid
from synthesizer.parametric import SFH, Stars
from synthesizer.particle import Stars as ParticleStars

grid = Grid("test_grid")
model = IncidentEmission(grid)

## Masses at the axis points: `from_sfzh`

`Stars.from_sfzh` takes the mass formed at each (age, metallicity) point of a pair of axes, e.g. an SFZH computed elsewhere on the grid's axes. Each point is a zero width bin, so on the same axes the SFZH is returned unchanged by the `sfzh` view.

(Passing `sfzh=` to `Stars` directly does the same thing but is deprecated in favour of `from_sfzh`.)

In [ ]:
sfzh = np.zeros((grid.log10ages.size, grid.metallicities.size))
sfzh[10, 5] = 1e9  # 1e9 Msun at a single age and metallicity
sfzh[20, 8] = 5e8

stars = Stars.from_sfzh(grid.log10ages, grid.metallicities, sfzh)
print(stars.bin_axes, stars.bin_masses.shape)
print(np.allclose(stars.sfzh, sfzh))

## Tabulated bins: `from_binned`

`Stars.from_binned` takes the bin edges along each axis and the mass formed in each bin. The edges can be anywhere: here a star formation history tabulated in three age bins (given as lookback times, so decreasing, which is accepted) with a single metallicity per bin. A single metallicity is a zero width bin, i.e. a repeated edge.

In [ ]:
age_edges = np.array([1000.0, 300.0, 50.0, 0.0]) * Myr  # oldest first
metallicity_edges = np.array([0.004, 0.004, 0.01, 0.01, 0.02, 0.02])

# The mass in each (age bin, metallicity bin): each age bin puts its mass at
# one of the metallicity points (the bins between the points are empty)
masses = np.zeros((3, 5))
masses[0, 0] = 5e9  # old stars at Z = 0.004
masses[1, 2] = 2e9  # intermediate stars at Z = 0.01
masses[2, 4] = 1e9  # young stars at Z = 0.02

binned = Stars.from_binned(
    grid.log10ages,
    grid.metallicities,
    age_edges,
    metallicity_edges,
    masses * Msun,
)
print(binned.initial_mass)

The `sfzh` attribute is a view of the bins mapped onto the object's axes (here the grid's), which is what `plot_sfzh` shows.

In [ ]:
binned.plot_sfzh(show=True)

Since the bins are spread uniformly, the emission is exactly that of particles spread uniformly through each bin. We can check that with a large number of particles.

In [ ]:
rng = np.random.default_rng(1)
n = 20000
ages, metals, part_masses = [], [], []
for (hi, lo), z, mass in zip(
    zip(age_edges[:-1].to("yr").value, age_edges[1:].to("yr").value),
    (0.004, 0.01, 0.02),
    masses.sum(axis=1),
):
    ages.append(lo + (hi - lo) * (np.arange(n) + 0.5) / n)
    metals.append(np.full(n, z))
    part_masses.append(np.full(n, mass / n))
particles = ParticleStars(
    initial_masses=np.concatenate(part_masses) * Msun,
    ages=np.concatenate(ages) * yr,
    metallicities=np.concatenate(metals),
)

binned_sed = binned.get_spectra(model)
particle_sed = particles.get_spectra(model)
ratio = (binned_sed.lnu / particle_sed.lnu).value
ok = particle_sed.lnu > 1e-10 * particle_sed.lnu.max()
print(f"Maximum relative difference: {np.max(np.abs(ratio[ok] - 1)):.1e}")

In [ ]:
fig, ax = plt.subplots()
ax.loglog(binned_sed.lam, binned_sed.lnu, label="Binned")
ax.loglog(particle_sed.lam, particle_sed.lnu, ls="--", label="Particles")
ax.set_xlim(1e2, 1e5)
ax.set_ylim(1e25, None)
ax.set_xlabel(r"$\lambda / \AA$")
ax.set_ylabel(r"$L_\nu / \mathrm{erg\,s^{-1}\,Hz^{-1}}$")
ax.legend()
plt.show()

## Semi-analytic models

The SAM loaders build their parametric galaxies with `from_binned`, so the SAM's bins are kept exactly rather than interpolated onto the grid:

- `load_SCSAM(fname, "parametric", grid)` reads an SC-SAM `sfhist` file, whose (age, metallicity) table goes in as is, with each tabulated metallicity a zero width bin.
- `load_SHARK(fname, grid, method="parametric")` reads a SHARK `star_formation_histories.hdf5` file. Each SHARK component (disks, merger driven bulges and disk instability driven bulges) becomes a named population of the same `Stars` (see the populations example), and each bin carries the metallicity of the stars formed in it as a zero width bin.

Both also offer a `"particle"` method, splitting each bin at the grid's ages into particles.

## Mass outside the grid

Mass outside the grid's axes is clamped onto the grid's edges when the emission is extracted, exactly as it is for particles. `get_fraction_outside_grid` reports how much of the mass this affects. A constant SFH from an age of zero has some of its mass younger than the youngest grid age:

In [ ]:
constant = Stars(
    grid.log10ages,
    grid.metallicities,
    sf_hist=SFH.Constant(max_age=100 * Myr),
    metal_dist=0.01,
    initial_mass=1e9 * Msun,
)
print(f"Youngest grid age: {10 ** grid.log10ages[0]:.1e} yr")
print(
    "Fraction outside the grid:",
    f"{constant.get_fraction_outside_grid(grid):.3f}",
)